In [1]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu \
                sentence-transformers transformers torch pypdf openpyxl pandas accelerate langchain-classic


In [2]:
import os
import glob
import pandas as pd
import torch

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_classic.memory import ConversationSummaryBufferMemory
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.documents import Document

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

/tmp/ipykernel_28368/1761552928.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
!pip install bitsandbytes
from transformers import BitsAndBytesConfig
import logging
import warnings

logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")

In [23]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quant_config,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model loaded on {device} (8-bit)")

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,          # shorter = less room to ramble/hallucinate
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
#llm to be aware of chat handler
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

llm_raw = HuggingFacePipeline(pipeline=hf_pipeline)
llm = ChatHuggingFace(llm=llm_raw, tokenizer=tokenizer)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded on cuda (8-bit)


In [24]:
pdf_paths = [
    "/content/sample.pdf",
    "/content/Mid-term Project.pdf",
]

excel_path = "/content/ai_and_rtl_knowledge_base.xlsx"

raw_documents = []

# ---- Load PDFs ----
for path in pdf_paths:
    if os.path.exists(path):
        loader = PyPDFLoader(path)
        pdf_docs = loader.load()
        for d in pdf_docs:
            d.metadata["source"] = os.path.basename(path)
        raw_documents.extend(pdf_docs)
        print(f"Loaded {len(pdf_docs)} page(s) from {path}")
    else:
        print(f"WARNING: {path} not found, skipping.")

# ---- Load Excel ("Google Sheets") ----
if os.path.exists(excel_path):
    sheets = pd.read_excel(excel_path, sheet_name=None)  # dict: {sheet_name: DataFrame}
    for sheet_name, df in sheets.items():
        df = df.fillna("")
        for _, row in df.iterrows():
            # Turn each row into a "column: value" text block
            content = "\n".join(f"{col}: {row[col]}" for col in df.columns)
            raw_documents.append(
                Document(
                    page_content=content,
                    metadata={"source": excel_path, "sheet": sheet_name}
                )
            )
    print(f"Loaded rows from sheets: {list(sheets.keys())}")
else:
    print(f"WARNING: {excel_path} not found, skipping.")

print(f"\nTotal raw documents collected: {len(raw_documents)}")


Loaded 4 page(s) from /content/sample.pdf
Loaded 2 page(s) from /content/Mid-term Project.pdf
Loaded rows from sheets: ['AI_Topics', 'RTL_Design']

Total raw documents collected: 20


In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Split into {len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("FAISS index built and retriever ready.")


Split into 25 chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built and retriever ready.


In [35]:
from langchain_core.prompts import PromptTemplate

summary_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=400,          # separate, larger budget so summaries don't get cut off
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
summary_llm_raw = HuggingFacePipeline(pipeline=summary_pipeline)
summary_llm = ChatHuggingFace(llm=summary_llm_raw, tokenizer=tokenizer)

custom_summary_prompt = PromptTemplate(
    input_variables=["summary", "new_lines"],
    template=(
        "You are updating a running summary of a conversation. "
        "Combine the existing summary with the new lines into one updated summary. "
        "Output ONLY the updated summary text itself — no labels, no headers, "
        "no phrases like 'summary:' or 'new summary', and do not repeat these instructions.\n\n"
        "Existing summary:\n{summary}\n\n"
        "New lines:\n{new_lines}\n\n"
        "Updated summary:"
    ),
)

memory = ConversationSummaryBufferMemory(
    llm=summary_llm,
    prompt=custom_summary_prompt,
    memory_key="chat_history",
    input_key="question",
    output_key="answer",
    max_token_limit=200,     # raised from 100: fewer summarization events = fewer chances to corrupt
    return_messages=True,
)

In [36]:
def show_memory_state():
    state = memory.load_memory_variables({})
    print("=" * 60)
    print("CURRENT MEMORY STATE (chat_history)")
    print("=" * 60)
    for msg in state.get("chat_history", []):
        role = msg.type.upper()
        print(f"[{role}] {msg.content}")

    # Estimate current raw buffer token count
    buffer_text = "\n".join(f"{m.type}: {m.content}" for m in state.get("chat_history", []))
    token_count = llm.get_num_tokens(buffer_text) if hasattr(llm, "get_num_tokens") else len(buffer_text.split())
    print(f"\nApprox. raw buffer tokens: {token_count}")
    print("=" * 60)


In [37]:
from langchain_core.prompts import PromptTemplate

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Answer the question using facts explicitly stated in the context below. "
        "You may perform simple reasoning, arithmetic, or comparisons using ONLY those "
        "stated facts (for example, calculating an age from a founding year, or comparing "
        "two listed items) — this is allowed and expected. "
        "Do NOT introduce any fact, number, date, or detail that is not stated in the context "
        "(for example, do not assume today's date, a number, or a fact not given to you). "
        "If a calculation requires information not provided in the context or question, "
        "say \"I don't know based on the provided knowledge base.\" "
        "If the question asks for a list of items (e.g. faculties, examples, steps), "
        "include ALL items found in the context, not just some. "
        "Otherwise, keep the answer concise (2-4 sentences).\n\n"
        "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    ),
)
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    output_key="answer",
    combine_docs_chain_kwargs={"prompt": qa_prompt},
    verbose=True,   # <-- prints the condensed standalone question + full prompt
)

def get_response(query):
    """Run one turn through the RAG + memory pipeline."""
    result = qa_chain.invoke({"question": query})
    answer = result["answer"].strip()
    sources = result.get("source_documents", [])
    return answer, sources


In [38]:
def print_sources(sources):
    if not sources:
        print("Sources: (none retrieved)")
        return
    print("Sources used:")
    for i, doc in enumerate(sources, 1):
        src = doc.metadata.get("source", "unknown")
        sheet = doc.metadata.get("sheet")
        label = f"{src}" + (f" [{sheet}]" if sheet else "")
        snippet = doc.page_content.replace("\n", " ")[:100]
        print(f"  {i}. {label} -> {snippet}...")


def chat():
    print("Chatbot started (type 'exit' to quit)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            break

        answer, sources = get_response(user_input)

        print("Bot:", answer)
        print_sources(sources)
        show_memory_state()


In [39]:
test_questions = [
    "What is Tips Hindawi University's founding year and motto?",       # PDF: sample.pdf
    "What faculties does it have?",                                     # follow-up, same doc
    "What is the mid-term project about?",                              # PDF: Mid-term_Project.pdf
    "What memory techniques does it mention I should use?",             # follow-up
    "What is the Wishbone Bus used for?",                               # Excel: RTL_Design sheet
    "Give me an example of unsupervised learning from the knowledge base.",  # Excel: AI_Topics sheet
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'#'*70}\nTurn {i}: {q}\n{'#'*70}")
    answer, sources = get_response(q)
    print("Bot:", answer)
    print_sources(sources)

show_memory_state()


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



######################################################################
Turn 1: What is Tips Hindawi University's founding year and motto?
######################################################################


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Answer the question using facts explicitly stated in the context below. You may perform simple reasoning, arithmetic, or comparisons using ONLY those stated facts (for example, calculating an age from a founding year, or comparing two listed items) — this is allowed and expected. Do NOT introduce any fact, number, date, or detail that is not stated in the context (for example, do not assume today's date, a number, or a fact not given to you). If a calculation requires information not provided in the context or question, say "I don't know based on the provided knowledge base." If the question asks for a list of items (e.g. faculties, examples, steps), include ALL items found i

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

> Finished chain.
Bot: Tips Hindawi University was founded in 1963 and its motto is "Knowledge, Integrity, Progress".
Sources used:
  1. sample.pdf -> 1. General Overview Tips Hindawi University (THU) is a premier institution of higher education locat...
  2. sample.pdf -> 9. Alumni and Impact Lina Darwish: UN Youth Ambassador Hassan Joudeh: CEO of ArabTech Noura Saleh: A...
  3. sample.pdf -> Vice President of Academic Affairs: Prof. Layla Mahmoud Dean of Students: Mr . Samer Hussein Registr...

######################################################################
Turn 2: What faculties does it have?
######################################################################


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: What is Tips Hindawi University's founding year and motto?
Assistant: 

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


> Finished chain.

> Finished chain.


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: The Wishbone Bus is used for standardizing communication between Intellectual Property (IP) cores in System-on-Chip (SoC) interconnects.
Sources used:
  1. /content/ai_and_rtl_knowledge_base.xlsx [RTL_Design] -> Category: RTL Design Concept: Wishbone Bus Description: An open-source hardware bus interface standa...
  2. /content/ai_and_rtl_knowledge_base.xlsx [RTL_Design] -> Category: RTL Design Concept: PWM Controller Description: Pulse-Width Modulation module used to cont...
  3. /content/ai_and_rtl_knowledge_base.xlsx [RTL_Design] -> Category: Digital Design Concept: Setup Time Description: Minimum time a data signal must be stable ...

######################################################################
Turn 6: Give me an example of unsupervised learning from the knowledge base.
######################################################################


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase t

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Answer the question using facts explicitly stated in the context below. You may perform simple reasoning, arithmetic, or comparisons using ONLY those stated facts (for example, calculating an age from a founding year, or comparing two listed items) — this is allowed and expected. Do NOT introduce any fact, number, date, or detail that is not stated in the context (for example, do not assume today's date, a number, or a fact not given to you). If a calculation requires information not provided in the context or question, say "I don't know based on the provided knowledge base." If the question asks for a list of items (e.g. faculties, examples, steps), include ALL items found in the context, not just some. Otherwise, keep the answer concise (2-4 sentences).

Context:
Topic: Machine Learning
Subtopic: Unsupervised Learning
Description: Models discover hidden pattern

In [32]:
show_memory_state()

CURRENT MEMORY STATE (chat_history)
[SYSTEM] Tips Hindawi University was founded in 1963 and its motto is "Knowledge, Integrity, Progress". However, the specific faculties it has include Faculty of Engineering, Faculty of Medicine and Health Sciences, Faculty of Business and Economics, Faculty of Arts and Humanities, Faculty of Law and International Studies, Faculty of Computer and Information Sciences, and Faculty of Architecture and Design. The mid-term project topic remains unclear.
[AI] Building a Basic AI Knowledge Assistant with Memory
[HUMAN] What memory techniques does it mention I should use?
[AI] Buffer memory (full chat history)  
Optional: Buffer Summary memory for long conversations
[HUMAN] What is the Wishbone Bus used for?
[AI] The Wishbone Bus is used for standardizing communication between IP cores.
[HUMAN] Give me an example of unsupervised learning from the knowledge base.
[AI] Customer segmentation (K-Means)

Approx. raw buffer tokens: 187


In [41]:
# How many years old is the university now if this year is 2026?
# Which of those faculties would a robotics student most likely enroll in?
#Does the project you just described actually require using the second memory technique you mentioned, or is it optional?
# Is the standard you just told me about used for software or hardware?
#Between the bus standard and the clustering example, which one belongs to hardware design and which to AI?

In [42]:
chat() #run interactively

Chatbot started (type 'exit' to quit)

You: How many years old is the university now if this year is 2026?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: Tips Hindawi University's founding year and motto were not specified initially, but it was founded in 1963 with the motto "Knowledge, Integrity, Progress". It has several faculties including Engineering, Medicine and Health Sciences, Business and Economics, Arts and Humanities, Law and International Studies, Computer and Information Sciences, and Architecture and Design.
Human: What is the mid-term project about?
Assistant: The mid-term project is about building a basic AI knowledge assistant with memory that can retrieve and answer questions from structured and unstructured knowledge sources, maintain conversational memory, and interact with multiple data sources such as PDFs and Google Sheets.
Human: What memory techniques does it mention I should use?
Assis

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Answer the question using facts explicitly stated in the context below. You may perform simple reasoning, arithmetic, or comparisons using ONLY those stated facts (for example, calculating an age from a founding year, or comparing two listed items) — this is allowed and expected. Do NOT introduce any fact, number, date, or detail that is not stated in the context (for example, do not assume today's date, a number, or a fact not given to you). If a calculation requires information not provided in the context or question, say "I don't know based on the provided knowledge base." If the question asks for a list of items (e.g. faculties, examples, steps), include ALL items found in the context, not just some. Otherwise, keep the answer concise (2-4 sentences).

Context:
1. General Overview
Tips Hindawi University (THU) is a premier institution of higher education loca

...



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: The mid-term project about building a basic AI knowledge assistant with memory that can retrieve and answer questions from structured and unstructured knowledge sources, maintain conversational memory, and interact with multiple data sources such as PDFs and Google Sheets was initiated by Hindawi University, founded in 1963 with the motto "Knowledge, Integrity, Progress", which has various faculties including Engineering, Medicine and Health Sciences, Business and Economics, Arts and Humanities, Law and International Studies, Computer and Information Sciences, and Architecture and Design. Buffer memory and optional buffer summary memory were mentioned for storing the full chat history in the AI project. The Wishbone Bus was not discussed in the provided contex

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Answer the question using facts explicitly stated in the context below. You may perform simple reasoning, arithmetic, or comparisons using ONLY those stated facts (for example, calculating an age from a founding year, or comparing two listed items) — this is allowed and expected. Do NOT introduce any fact, number, date, or detail that is not stated in the context (for example, do not assume today's date, a number, or a fact not given to you). If a calculation requires information not provided in the context or question, say "I don't know based on the provided knowledge base." If the question asks for a list of items (e.g. faculties, examples, steps), include ALL items found in the context, not just some. Otherwise, keep the answer concise (2-4 sentences).

Context:
3. Academic Structure
3.1 Faculties
Faculty of Engineering
Faculty of Medicine and Health Sciences


[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

> Finished chain.
Bot: A robotics student would most likely enroll in the Faculty of Computer and Information Sciences, as it is one of the faculties that includes Computer and Information Sciences.
Sources used:
  1. sample.pdf -> 3. Academic Structure 3.1 Faculties Faculty of Engineering Faculty of Medicine and Health Sciences F...
  2. sample.pdf -> Vice President of Academic Affairs: Prof. Layla Mahmoud Dean of Students: Mr . Samer Hussein Registr...
  3. sample.pdf -> Robotics Club Arabic Calligraphy Circle Environmental Awareness Network International Students Union...
CURRENT MEMORY STATE (chat_history)
[SYSTEM] The mid-term project about building a basic AI knowledge assistant with memory that can retrieve and answer questions from structured and unstructured knowledge sources, maintain conversational memory, and interact with multiple data sources such as PDFs and Google Sheets was initiated by Hindawi University, founded in 1963 with the motto "Knowledge, 